In [1]:
from IPython.display import IFrame

IFrame("FD_nine_pt_stencil.pdf", width=1000, height=800)

## Answer - 7

In [2]:
import numpy as np

# Matrix D1 for the internal node of dimension N\times N
def matrix_D1(M):
    N = M-1
    D1 = np.zeros((N,N))
    for i in range(N):
        D1[i, i] = 4
        if i != 0:
            D1[i, i-1] = 1
        if i != N-1:
            D1[i, i+1] = 1
    return D1

# Matrix D2 for the internal node of dimension N\times N
def matrix_D2(M):
    N = M-1
    D2 = np.zeros((N,N))
    for i in range(N):
        D2[i, i] = 20
        if i != 0:
            D2[i, i-1] = -4
        if i != N-1:
            D2[i, i+1] = -4
    return D2

# Matrix of the formulation [[0, -1], [-1, 0]]
def create_tridiagonal_matrix(M):
    N = M-1
    A = np.zeros((N, N))
    for i in range(N):
        if i != 0:
            A[i, i-1] = -1
        if i != N-1:
            A[i, i+1] = -1
    return A

# Right hand side
def f_rhs(x, y):
    return -( 12*x**2*y**5 + 20*x**4*y**3 + 17*np.sin(x*y)*(x**2 + y**2) )

def del_f_rhs(x,y):
    return (-120*x**4*y -480*x**2*y**3 -24*y**5 -136*x*y*np.cos(x*y) -68*np.sin(x*y)+ 17*x**4*np.sin(x*y)+34*x**2*y**2*np.sin(x*y) 
            +17*y**4*np.sin(x*y) )
    
def f_cross(x,y,h):
    return f_rhs(x, y) + (h**2/12)*del_f_rhs(x,y)

# Exact Solution
def exact_solution(x, y):
    return x**4*y**5-17*np.sin(x*y)

# Solve the Poisson Problem
def solve_poisson(M):
    # Interval points
    a, b = 0, 1
    h = (b - a) / M
    N = M - 1  # number of interior points

    D1 = matrix_D1(M)
    D2 = matrix_D2(M)
    I = np.eye(N)
    # Diagonal part of matrix A
    A1 = np.kron(I, D2)
    # Other part of matrix A
    A2 = np.kron(create_tridiagonal_matrix(M), D1)
    A = (A1 + A2)
    # Load vector and Dirichlet BCs
    F = np.zeros(N * N)
    for i in range(N):
        for j in range(N):
            x = (i+1)*h
            y = (j+1)*h
            l = i*N + j
            F[l] = 6*h**2*f_cross(x,y,h)

            # Boundary contributions 
            # Left boundary (x=0)
            if i == 0:
                F[l] += exact_solution(0, y - h) + 4 * exact_solution(0, y) + exact_solution(0, y + h)
            # Right boundary (x=1)
            if i == N - 1:
                F[l] += exact_solution(1, y - h) + 4 * exact_solution(1, y) + exact_solution(1, y + h)
            # Bottom boundary (y=0)
            if j == 0:
                F[l] += exact_solution(x - h, 0) + 4 * exact_solution(x, 0) + exact_solution(x + h, 0)
            # Top boundary (y=1)
            if j == N - 1:
                F[l] += exact_solution(x - h, 1) + 4 * exact_solution(x, 1) + exact_solution(x + h, 1)
            
            # Removing points double-counted above
            if i == 0 and j == 0:
                F[l] -= exact_solution(0, 0) 
            if i == 0 and j == N - 1:
                F[l] -= exact_solution(0, 1)
            if i == N - 1 and j == 0:
                F[l] -= exact_solution(1, 0)
            if i == N - 1 and j == N - 1:
                F[l] -= exact_solution(1, 1)
                
    # Solve system
    U_vec = np.linalg.solve(A, F)

    # Reshape to 2D grid
    U_grid = U_vec.reshape((N, N))

    # Exact solution
    U_exact = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            x = (i+1)*h
            y = (j+1)*h
            U_exact[i, j] = exact_solution(x, y)

    # Infinity norm of error
    error_inf = np.max(np.abs(U_grid - U_exact))

    return h, error_inf

# Mesh sizes to test
mesh_sizes = [4, 8, 16, 32, 64, 128]
error = []
step_size = []

for M in mesh_sizes:
    h, err_inf= solve_poisson(M)
    step_size.append(h)
    error.append(err_inf)

for i in range(len(step_size)):
    if i == 0:
        eoc = 0
    else:
        eoc = np.log(error[i-1] / error[i]) / np.log(step_size[i-1] / step_size[i])
    print(f"h : {step_size[i]:.5f}, Error : {error[i]:.9f}, EOC : {eoc:.5f}")

h : 0.25000, Error : 0.001070910, EOC : 0.00000
h : 0.12500, Error : 0.000071717, EOC : 3.90039
h : 0.06250, Error : 0.000004531, EOC : 3.98456
h : 0.03125, Error : 0.000000285, EOC : 3.99164
h : 0.01562, Error : 0.000000018, EOC : 3.99903
h : 0.00781, Error : 0.000000001, EOC : 3.99986


### As we can see from above results that :

### Order of convergence $\approx$ 4